# 4. Compile LLM Labelled Corpus Results

This notebook compiles parsed LLM batch/direct outputs from `llm_labelled_data` into corpus-level labelled data files used by the downstream name-processing and network notebooks.


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


from pathlib import Path

# Code is run from the OneDrive project code folder.
CODE_ROOT = REPO_ROOT
TEXT_MINING_CODE_ROOT = CODE_ROOT / "text_mining"

# Data is read from and written to the share-drive project folder.
SHARE_ROOT = DATA_ROOT
# DATA_ROOT is configured through project_config.
ROOT = DATA_ROOT
OUTPUT_ROOT = ROOT / "llm_labelled_data"
BATCH_RUNS_DIR = OUTPUT_ROOT / "batch_runs"
DIRECT_RUNS_DIR = OUTPUT_ROOT / "direct_runs"
ROUTING_DIR = OUTPUT_ROOT / "routing"
COMPILED_DIR = OUTPUT_ROOT / "compiled"

FILE_MANIFEST_FILE = ROUTING_DIR / "filtered_texts_file_manifest.csv"

ALL_ROWS_FILE = COMPILED_DIR / "llm_labelled_results_all_rows.csv"
MASTER_FILE = COMPILED_DIR / "llm_labelled_results_master.csv"
LONG_SOURCES_FILE = COMPILED_DIR / "llm_labelled_data_sources_long.csv"
DUPLICATES_FILE = COMPILED_DIR / "llm_labelled_duplicate_rows_audit.csv"
COVERAGE_BY_SUBFOLDER_FILE = COMPILED_DIR / "llm_labelled_coverage_by_subfolder.csv"
SUMMARY_FILE = COMPILED_DIR / "compile_summary.json"

COMPILED_DIR.mkdir(parents=True, exist_ok=True)


def utc_iso_from_timestamp(ts):
    return datetime.fromtimestamp(ts, timezone.utc).isoformat()


def infer_run_type(results_file):
    parts = {part.lower() for part in results_file.parts}
    if "batch_runs" in parts:
        return "batch"
    if "direct_runs" in parts:
        return "direct"
    return "unknown"


def read_json_if_exists(path):
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return {}


## 1. Discover run-level results files

The compiler deliberately uses only run-level files named `results.csv`. That avoids accidentally double-counting smaller intermediate artifacts if a direct run was restarted or partially written.


In [ ]:
result_files = sorted(BATCH_RUNS_DIR.glob("*/results.csv")) + sorted(DIRECT_RUNS_DIR.glob("*/results.csv"))

file_rows = []
for results_file in result_files:
    run_dir = results_file.parent
    run_type = infer_run_type(results_file)
    info = read_json_if_exists(run_dir / "batch_info.json")

    file_rows.append(
        {
            "run_type": run_type,
            "run_name": run_dir.name,
            "results_file": str(results_file),
            "result_file_mtime_utc": utc_iso_from_timestamp(results_file.stat().st_mtime),
            "plan_batch_id": info.get("plan_batch_id"),
            "openai_batch_id": info.get("batch_id"),
            "batch_status": info.get("status"),
            "n_requests": info.get("n_requests"),
            "submitted_at_utc": info.get("submitted_at_utc"),
            "completed_at": info.get("completed_at"),
        }
    )

results_inventory = pd.DataFrame(file_rows)
results_inventory


## 2. Load and combine all rows

This creates a raw combined table. It preserves duplicates, because duplicates are useful for auditing reruns and notebook mistakes.


In [ ]:
frames = []

for row in file_rows:
    results_file = Path(row["results_file"])
    df = pd.read_csv(results_file)

    for key, value in row.items():
        if key != "results_file":
            df[key] = value
    df["results_file"] = row["results_file"]

    frames.append(df)

if frames:
    all_rows = pd.concat(frames, ignore_index=True, sort=False)
else:
    all_rows = pd.DataFrame()

if not all_rows.empty:
    all_rows["global_index"] = pd.to_numeric(all_rows["global_index"], errors="coerce").astype("Int64")
    all_rows["is_success"] = all_rows["status"].fillna("").str.lower().eq("ok")
    all_rows["result_file_mtime_utc"] = pd.to_datetime(all_rows["result_file_mtime_utc"], utc=True, errors="coerce")
    all_rows = all_rows.sort_values(["global_index", "result_file_mtime_utc", "run_name"], na_position="last")

all_rows.to_csv(ALL_ROWS_FILE, index=False, encoding="utf-8")

print(f"Loaded {len(result_files):,} run-level results files.")
print(f"Raw combined rows: {len(all_rows):,}")
print(f"Saved: {ALL_ROWS_FILE}")
all_rows.head()


## 3. Deduplicate into a master corpus file

Deduplication key:
- Prefer `global_index` when present.
- Fall back to `relative_path` only if `global_index` is missing.

Selection rule:
- Prefer successful rows over failed rows.
- Within the same success/failure status, keep the newest run-level `results.csv`.


In [ ]:
if all_rows.empty:
    master = all_rows.copy()
    duplicate_rows = all_rows.copy()
else:
    all_rows["dedupe_key"] = all_rows["global_index"].astype("string")
    missing_key = all_rows["dedupe_key"].isna() | all_rows["dedupe_key"].eq("<NA>")
    all_rows.loc[missing_key, "dedupe_key"] = all_rows.loc[missing_key, "relative_path"].astype("string")

    duplicate_mask = all_rows.duplicated("dedupe_key", keep=False)
    duplicate_rows = all_rows.loc[duplicate_mask].sort_values(["dedupe_key", "result_file_mtime_utc", "run_name"])

    ranked = all_rows.sort_values(
        ["dedupe_key", "is_success", "result_file_mtime_utc", "run_name"],
        ascending=[True, True, True, True],
        na_position="first",
    )
    master = ranked.groupby("dedupe_key", as_index=False, sort=False).tail(1)
    master = master.drop(columns=["dedupe_key"], errors="ignore")
    master = master.sort_values(["global_index", "relative_path"], na_position="last").reset_index(drop=True)

duplicate_rows.to_csv(DUPLICATES_FILE, index=False, encoding="utf-8")
master.to_csv(MASTER_FILE, index=False, encoding="utf-8")

print(f"Duplicate raw rows: {len(duplicate_rows):,}")
print(f"Master rows: {len(master):,}")
print(f"Saved master: {MASTER_FILE}")
print(f"Saved duplicate audit: {DUPLICATES_FILE}")
master.head()


In [ ]:
master[["file_name","n_data_sources", "reason_short"]]

## 4. Create a long data-source table

This is usually the most practical file for downstream analysis: every extracted source name becomes its own row.


In [ ]:
def parse_sources(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if not text or text == "[]":
        return []
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return [text]
    if isinstance(parsed, list):
        return [str(item) for item in parsed if str(item).strip()]
    if isinstance(parsed, str) and parsed.strip():
        return [parsed]
    return []


long_rows = []
for _, row in master.iterrows():
    sources = parse_sources(row.get("data_sources_json"))
    for source_order, source_name in enumerate(sources, start=1):
        long_rows.append(
            {
                "global_index": row.get("global_index"),
                "relative_path": row.get("relative_path"),
                "source_subfolder": row.get("source_subfolder"),
                "file_name": row.get("file_name"),
                "db_use": row.get("db_use"),
                "data_source": source_name,
                "source_order": source_order,
                "reason_short": row.get("reason_short"),
                "run_type": row.get("run_type"),
                "run_name": row.get("run_name"),
                "prompt_revision": row.get("prompt_revision"),
                "status": row.get("status"),
            }
        )

long_sources = pd.DataFrame(long_rows)
long_sources.to_csv(LONG_SOURCES_FILE, index=False, encoding="utf-8")

print(f"Long data-source rows: {len(long_sources):,}")
print(f"Saved: {LONG_SOURCES_FILE}")
long_sources.head()


## 5. Coverage and summary

This checks how much of the full `filtered_texts` manifest has compiled labels so far. The manifest is large, so the notebook only saves a compact coverage summary by source subfolder.


In [ ]:
if FILE_MANIFEST_FILE.exists() and not master.empty:
    manifest_cols = ["global_index", "relative_path", "source_subfolder"]
    manifest = pd.read_csv(FILE_MANIFEST_FILE, usecols=manifest_cols)
    manifest["global_index"] = pd.to_numeric(manifest["global_index"], errors="coerce").astype("Int64")

    labelled_keys = set(master["global_index"].dropna().astype(int).tolist())
    manifest["has_llm_label"] = manifest["global_index"].isin(labelled_keys)

    coverage_by_subfolder = (
        manifest.groupby("source_subfolder", dropna=False)["has_llm_label"]
        .agg(total_files="size", labelled_files="sum")
        .reset_index()
    )
    coverage_by_subfolder["unlabelled_files"] = (
        coverage_by_subfolder["total_files"] - coverage_by_subfolder["labelled_files"]
    )
    coverage_by_subfolder["labelled_share"] = (
        coverage_by_subfolder["labelled_files"] / coverage_by_subfolder["total_files"]
    )
    coverage_by_subfolder.to_csv(COVERAGE_BY_SUBFOLDER_FILE, index=False, encoding="utf-8")
else:
    coverage_by_subfolder = pd.DataFrame()

summary = {
    "compiled_at_utc": datetime.now(timezone.utc).isoformat(),
    "n_result_files": int(len(result_files)),
    "n_raw_rows": int(len(all_rows)),
    "n_master_rows": int(len(master)),
    "n_duplicate_raw_rows": int(len(duplicate_rows)),
    "n_db_use_1": int((pd.to_numeric(master.get("db_use", pd.Series(dtype=float)), errors="coerce") == 1).sum()) if not master.empty else 0,
    "n_long_data_source_rows": int(len(long_sources)),
    "input_tokens": int(pd.to_numeric(master.get("input_tokens", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()) if not master.empty else 0,
    "output_tokens": int(pd.to_numeric(master.get("output_tokens", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()) if not master.empty else 0,
    "total_tokens": int(pd.to_numeric(master.get("total_tokens", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()) if not master.empty else 0,
    "files": {
        "all_rows": str(ALL_ROWS_FILE),
        "master": str(MASTER_FILE),
        "long_sources": str(LONG_SOURCES_FILE),
        "duplicates": str(DUPLICATES_FILE),
        "coverage_by_subfolder": str(COVERAGE_BY_SUBFOLDER_FILE),
    },
}

SUMMARY_FILE.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(summary, indent=2, ensure_ascii=False))
coverage_by_subfolder
